# Variable-Channel Spatial ASR — 재현 노트북

마이크가 늘어날 때 공간 정보가 음성인식의 화자 구분을 돕는지 검증하는 가변 채널 벤치마크입니다.
여러 화자가 번갈아 말하는 **본 평가**(구축 중)에 앞서, 화자가 한 명인 **사전 측정**으로
마이크 수별로 얻을 수 있는 공간 정보의 양을 쟀습니다.

이 노트북은 저장소에 커밋된 사전 측정 숫자 중 **Baseline 2(`results/baseline2.json`)가 재현된다**는 것을 보여줍니다.
모든 셀이 GPU 없이 무료 CPU 런타임에서 돌아갑니다. 채점기는 표준 라이브러리만 쓰고,
베이스라인 재실행에만 `transformers`와 동결 ASR 체크포인트가 필요합니다.

저장소: https://github.com/hd1534/variable-channel-spatial-asr

## 1. 저장소와 데이터 받기

데이터셋이 저장소 안에 있어서 clone 한 번이면 끝납니다 (약 360 MB).

In [ ]:
!git clone --depth 1 https://github.com/hd1534/variable-channel-spatial-asr.git
%cd variable-channel-spatial-asr
!ls -la data/eval-dev

## 2. 무결성 검증

`manifest.json`이 파일마다 SHA256을 들고 있습니다. 받은 게 생성된 것과 같은지
직접 대조합니다. 하나라도 어긋나면 아래 숫자를 믿으면 안 됩니다.

In [ ]:
import hashlib, json, os

m = json.load(open("data/eval-dev/manifest.json", encoding="utf-8"))
files = m["files"]
items = list(files.items()) if isinstance(files, dict) else [(f["path"], f) for f in files]

bad = missing = checked = 0
total = 0
for path, rec in items:
    want = rec if isinstance(rec, str) else (rec.get("sha256") or rec.get("hash"))
    full = os.path.join("data/eval-dev", path)
    if not os.path.isfile(full):
        # RIR 번들은 의도적으로 커밋하지 않는다 (채점이 읽지 않는 생성 내부물)
        if "/rir/" in path or path.startswith("rir/"):
            continue
        missing += 1
        continue
    h = hashlib.sha256()
    with open(full, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    total += os.path.getsize(full)
    checked += 1
    if want and h.hexdigest() != want:
        bad += 1
        print("  해시 불일치:", path)

print(f"검증 {checked}개 / {total/1024/1024:.1f} MB")
print(f"불일치 {bad}개, 누락 {missing}개")
print()
print("official        :", m.get("official"))
print("차단 사유       :", m.get("official_blockers") or "없음")
print("시드            :", m.get("seed"))
print("RIR 엔진        :", m.get("rir_engine"))
print("음원            :", m.get("source_mode"))
assert bad == 0, "해시가 어긋났다. 다시 clone 하라."

## 3. 채점기 검증 — GPU도 의존성도 없이

`evaluate.py`는 **표준 라이브러리만으로** 동작합니다. numpy는 선택적 가속일 뿐이고,
numpy를 막아도 테스트가 전부 통과합니다. 채점 결과를 재현하는 데 아무 설치도
필요 없다는 뜻입니다.

In [ ]:
!python3 -m unittest discover -s tests 2>&1 | tail -4

In [ ]:
# numpy 를 실제로 막고 다시 돌린다 -- 순수 파이썬 경로가 살아 있는지 확인
import importlib.abc, io, subprocess, sys, textwrap

probe = textwrap.dedent('''
    import importlib.abc, io, sys, unittest
    class Block(importlib.abc.MetaPathFinder):
        def find_spec(self, name, path=None, target=None):
            if name == "numpy" or name.startswith("numpy."):
                raise ImportError("numpy blocked")
            return None
    sys.meta_path.insert(0, Block())
    sys.path[:0] = [".", "tests"]
    import evaluate, test_evaluate
    print("HAS_NUMPY =", evaluate.HAS_NUMPY)
    r = unittest.TextTestRunner(stream=io.StringIO()).run(
        unittest.TestLoader().loadTestsFromModule(test_evaluate))
    print(f"{r.testsRun} tests, {len(r.failures)} failures, {len(r.errors)} errors")
''')
print(subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True).stdout)

## 4. 커밋된 사전 측정 숫자 재현 (Baseline 2)

`results/baseline2.json`에 들어 있는 숫자가 이 데이터에서 실제로 나오는지 확인합니다.
베이스라인은 동결 ASR(`facebook/wav2vec2-base-960h`)을 내려받지 않고 로컬 캐시에서만 불러오므로,
아래에서 체크포인트를 먼저 한 번 받아 둡니다 (약 360 MB).

- `LIMIT = 48` — 몇 분. 파이프라인이 도는지만 봅니다 (숫자는 부분집합이라 커밋값과 다름)
- `LIMIT = None` — 864케이스 전부. CPU로 10분 안팎. 커밋된 숫자는 macOS(Apple Silicon)에서 만든 값이라,
  같은 환경에서는 실행마다 달라지는 rtf(처리 속도)를 뺀 13개 지표가 비트 단위로 일치하고,
  Colab(Linux x86-64)에서는 마지막 자릿수가 다를 수 있습니다

In [ ]:
LIMIT = 48   # 전체 재현은 None 으로 바꾼다

In [ ]:
!pip -q install "transformers==4.46.3" "soundfile==0.13.1" 2>&1 | tail -2

In [ ]:
# 베이스라인은 체크포인트를 직접 받지 않는다 (오프라인 고정).
# 새 런타임이면 여기서 한 번 받아 캐시에 둔다.
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

ASR = "facebook/wav2vec2-base-960h"
Wav2Vec2Processor.from_pretrained(ASR)
Wav2Vec2ForCTC.from_pretrained(ASR)
print("체크포인트 캐시 완료:", ASR)

In [ ]:
import subprocess, sys

# --track 1 은 현재 코드(spec.py)에서 단일 화자(사전 측정) 채점을 가리키는 값이다.
cmd = [sys.executable, "-m", "baselines.dsp_srp_phat",
       "--eval-dir", "data/eval-dev", "--out", "runs/b2.jsonl",
       "--manifest", "data/eval-dev/meta.jsonl", "--scope", "causal", "--track", "1"]
if LIMIT:
    cmd += ["--limit", str(LIMIT)]

import os; os.makedirs("runs", exist_ok=True)
res = subprocess.run(cmd, capture_output=True, text=True)
# 베이스라인은 진행 줄과 마지막 요약 줄을 stderr 로 쓴다 (stdout 은 비어 있다)
if res.returncode != 0:
    print(res.stderr[-1500:])
    raise RuntimeError("베이스라인 실행이 실패했다. 위 오류를 확인하라.")
print(res.stderr[-600:])

`--limit` 를 쓸 때는 **정답도 같이 잘라야 합니다.** 안 그러면 예측하지 않은 케이스가
전부 결측으로 처리되어 NLL 이 페널티 바닥(27.63)까지 치솟습니다. 평가기는 정답에 있는
모든 케이스를 채점하도록 설계돼 있고, 그건 "답하기 싫은 시점을 비워두는" 전략을
막기 위한 규칙입니다.

In [ ]:
import json

GT = "data/eval-dev/ground_truth.jsonl"
if LIMIT:
    have = {json.loads(l)["case_id"] for l in open("runs/b2.jsonl", encoding="utf-8")}
    rows = [l for l in open(GT, encoding="utf-8") if json.loads(l)["case_id"] in have]
    GT = "runs/gt_subset.jsonl"
    open(GT, "w", encoding="utf-8").writelines(rows)
    print(f"정답을 {len(rows)}케이스로 축소 (예측과 일치시킴)")
else:
    print("전체 864케이스로 채점한다")

In [ ]:
# --track 1 은 현재 코드(spec.py)에서 단일 화자(사전 측정) 채점을 가리키는 값이다.
!python3 evaluate.py --pred runs/b2.jsonl --gt $GT --track 1 --out runs/b2.scores.json > /dev/null

import json
got = json.load(open("runs/b2.scores.json"))["metrics"]
ref = json.load(open("results/baseline2.json"))["metrics"]

print(f"{'지표':28s}{'재현':>12s}{'커밋됨':>12s}")
print("-" * 52)
for k in ("wer", "azimuth_mae_deg", "azimuth_acc_10deg", "distance_mae_m",
          "pos_error_2d_m", "azimuth_nll", "distance_nll",
          "azimuth_interval_width_deg", "rtf"):
    f = lambda v: "N/A" if v is None else f"{v:.4f}"
    print(f"{k:28s}{f(got.get(k)):>12s}{f(ref.get(k)):>12s}")

if not LIMIT:
    import math
    # 커밋값은 macOS(Apple Silicon)에서 만든 값이다. 같은 환경이면 비트 단위로 같고,
    # Colab(Linux x86-64)처럼 FFT·행렬곱·수학 함수 구현이 다르면 마지막 자릿수가 달라질 수 있다.
    # rtf 는 실행 기기의 처리 속도 측정치라 같은 기기에서도 부하에 따라 30% 가량
    # 흔들리므로(실측 0.0176 -> 0.0229) 동일성 검사에서 빼고 따로 보여준다.
    TIMING = {"rtf"}
    close = lambda a, b: (isinstance(a, float) and isinstance(b, float)
                          and math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-12))
    checked = [k for k in ref if ref[k] is not None and k not in TIMING]
    exact = [k for k in checked if got[k] == ref[k]]
    near = [k for k in checked if k not in exact and close(got[k], ref[k])]
    differ = [k for k in checked if k not in exact and k not in near]
    print()
    print(f"결정적 지표 {len(checked)}개: 비트 단위 동일 {len(exact)}개, "
          f"반올림 수준 차이(상대 1e-9 이내) {len(near)}개, 그보다 큰 차이 {len(differ)}개")
    for k in near + differ:
        print(f"  {'근사' if k in near else '다름'}  {k}: 재현 {got[k]!r} / 커밋 {ref[k]!r}")
    print(f"rtf (기기·부하 의존이라 비교 제외): 재현 {got['rtf']:.4f} / 커밋 {ref['rtf']:.4f}")
else:
    print()
    print(f"LIMIT={LIMIT} 부분집합이다. 분포 지표(NLL, 구간폭)는 전체값에 가깝게 나오고")
    print("WER·MAE·거리·2D 오차는 부분집합의 발화와 방위·거리 분포가 전체와 달라서 다르다. 전체 비교는 LIMIT=None.")

## 5. 사전 측정 — 마이크 수별 공간 정보량

방위 정확도를 마이크 수별로 봅니다. 방위 추정은 **M=2에서 M=3으로 갈 때** 크게 좋아집니다 (53.12° → 9.51°).

마이크 2개는 두 마이크를 잇는 축 양쪽의 시간차가 같아서 앞뒤를 구분하지 못합니다.
예측 분포가 앞뒤 두 봉우리로 갈려 점 추정 오차가 큽니다. 비공선 세 번째 마이크가 그 모호성을 깹니다.

아래 표는 4절에서 만든 `runs/b2.jsonl`을 읽습니다. 위 수치(53.12° → 9.51°)는 `LIMIT = None`으로
864케이스를 모두 돌렸을 때의 값이고, 기본값 `LIMIT = 48`에서는 부분집합 값이 나옵니다.

다만 화자 구분에는 정확한 방향이 아니라 두 화자의 공간 신호가 다르기만 하면 됩니다.
그래서 화자 구분은 방위 추정과 달리 **M=1에서 M=2로 갈 때** 먼저 크게 좋아질 것으로 예상하며(가설 H1),
이는 다화자 본 평가에서 검증합니다.

In [ ]:
import json, math, sys, collections, statistics as st
sys.path.insert(0, ".")
import spec

truth = {}
for line in open("data/eval-dev/ground_truth.jsonl", encoding="utf-8"):
    r = json.loads(line)
    truth[r["case_id"]] = {f["eval_time_sec"]: f["azimuth_deg"] for f in r.get("frames", [])}

# 집계는 spec.SPATIAL_AGGREGATION 과 같아야 한다 -- 케이스 안에서 먼저 평균을 내고
# 케이스끼리 동일 가중으로 합친다. 시점을 한 덩어리로 평균하면 발화가 긴 케이스가
# 더 무거워져서 공식 점수와 어긋난다.
per_case = collections.defaultdict(lambda: collections.defaultdict(list))
for line in open("runs/b2.jsonl", encoding="utf-8"):
    r = json.loads(line)
    if r.get("record_type") != spec.RECORD_TYPE_TIMEPOINT:
        continue
    t = truth.get(r["case_id"], {}).get(r["eval_time_sec"])
    if t is None:
        continue
    d = per_case[r["case_id"]]
    d["err"].append(spec.circular_error_deg(t, r["pred_azimuth_deg"]))
    p = r.get("azimuth_probs")
    if p:
        d["nll"].append(-math.log(max(p[spec.azimuth_bin_index(t)], spec.NLL_EPSILON)))

by_m = collections.defaultdict(lambda: collections.defaultdict(list))
for case_id, d in per_case.items():
    M = spec.parse_case_id(case_id)[1].count("1")
    for k, v in d.items():
        if v:
            by_m[M][k].append(st.mean(v))

print(f"{'마이크 수':>10s}{'Azimuth MAE':>14s}{'NLL':>9s}{'유효 bin':>10s}")
print("-" * 44)
for M in sorted(by_m):
    a = st.mean(by_m[M]["nll"])
    print(f"{M:>10d}{st.mean(by_m[M]['err']):13.2f}°{a:9.4f}{math.exp(a):10.2f}")
print()
overall = st.mean([st.mean(d["err"]) for d in per_case.values()])
scored = json.load(open("runs/b2.scores.json", encoding="utf-8"))["metrics"]["azimuth_mae_deg"]
committed = json.load(open("results/baseline2.json", encoding="utf-8"))["metrics"]["azimuth_mae_deg"]
print(f"케이스 macro 평균 {overall:.4f}° -- 4절에서 같은 예측을 채점한 evaluate.py 값({scored:.4f}°)과 같아야 한다")
if LIMIT:
    print(f"LIMIT={LIMIT} 부분집합이다. 커밋값({committed:.4f}°)과 53.12° → 9.51° 는 LIMIT=None 으로 전체를 돌려야 나온다")
else:
    print(f"커밋값 {committed:.4f}° (macOS 에서 만든 값이라 Colab 에서는 마지막 자릿수가 다를 수 있다)")
print()
print(f"균일 분포 기준선: NLL = ln 36 = {math.log(36):.4f}, 유효 bin 36, 우연 MAE 90°")

## 6. 다음 단계 — 다화자 본 평가

위 숫자는 모두 화자가 한 명인 사전 측정입니다. 본 평가는 같은 방 서로 다른 위치의 여러 화자가
번갈아 말하는 조건에서, 공간 단서를 쓰는 시스템이 마이크 수별로 목소리 전용 기준선보다
화자 구분 주 지표(`cpWER`·`Spk Acc`)를 개선하는지 검증합니다.

| 구성 | 내용 | 상태 |
|---|---|---|
| 다화자 평가셋 | 같은 방 서로 다른 위치의 화자들이 번갈아 말하는 세션 | 구축 중 |
| 비교 시스템 | 목소리 전용(ECAPA-TDNN + 온라인 군집), 공간 전용(SRP-PHAT 방위 군집), 단순 결합(두 점수 합산) | 구축 예정 |
| 제안 모델 | `TAC + ECAPA + GRU`. 가변 채널 TAC 공간 특징과 화자 임베딩을 결합하는 인과 GRU로, 청크 상태 초기화(`Chunk-Reset`)와 유지(`Carry-Over`)를 비교 | 12월 7일까지 학습 |

현재 채점 계약(`spec.py` v0.2.0)은 세션당 2인까지 채점하므로, 3인 이상은 화자 수 범위가 정해지면
다화자 평가셋과 함께 계약을 개정해 지원할 예정입니다.

거리는 잔향 때문에 단서가 약해 목표로 두지 않습니다. 4절 표 '커밋됨' 열의 거리 수치(`distance_nll` = ln 8,
`distance_mae_m` 1.101 m)는 두 베이스라인이 거리를 추정하지 않고 8구간 균일 분포와 그 중앙값(1.9052 m)을 낸 무정보 기준값입니다.
외부 참조로 검토한 `Qwen3-ASR-1.7B`는 공식 문서상 화자 구분 기능이 없어 비교에서 뺐습니다.